In [122]:
import pandas as pd

df = pd.read_excel("../data/Online Retail.xlsx")

In [123]:
# Odd looking stock codes

non_product_codes = {
    "POST", "DOT", "M", "m", "AMAZONFEE", "S", "B"
}

In [124]:
# We choose to consider the following cleaned dataset: 
# keep positive values (unit prices + Quantity) — this excludes cancellations + zero-price exceptional transactions; 
# removing odd looking stock codes;
# keeping UK transactions only 

df_clean = df[
    (df["Quantity"] > 0)
    & (df["UnitPrice"] > 0) 
    & ~(df["StockCode"].astype(str).isin(non_product_codes))
    & (df["Country"] == "United Kingdom")
].copy()

In [125]:
# Assign each transaction to a week

df_clean["Week"] = df_clean["InvoiceDate"].dt.to_period("W")

In [126]:
df_clean

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Week
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,2010-11-29/2010-12-05
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,2010-11-29/2010-12-05
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,2010-11-29/2010-12-05
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,2010-11-29/2010-12-05
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,2010-11-29/2010-12-05
...,...,...,...,...,...,...,...,...,...
541889,581585,22466,FAIRY TALE COTTAGE NIGHT LIGHT,12,2011-12-09 12:31:00,1.95,15804.0,United Kingdom,2011-12-05/2011-12-11
541890,581586,22061,LARGE CAKE STAND HANGING STRAWBERY,8,2011-12-09 12:49:00,2.95,13113.0,United Kingdom,2011-12-05/2011-12-11
541891,581586,23275,SET OF 3 HANGING OWLS OLLIE BEAK,24,2011-12-09 12:49:00,1.25,13113.0,United Kingdom,2011-12-05/2011-12-11
541892,581586,21217,RED RETROSPOT ROUND CAKE TINS,24,2011-12-09 12:49:00,8.95,13113.0,United Kingdom,2011-12-05/2011-12-11


In [127]:
# the first and last weeks and last weeks are incomplete

print(df_clean["InvoiceDate"].min(), df_clean["Week"].min())
print(df_clean["InvoiceDate"].max(), df_clean["Week"].max())

2010-12-01 08:26:00 2010-11-29/2010-12-05
2011-12-09 12:49:00 2011-12-05/2011-12-11


In [128]:
# remove the first and last incomplete calendar weeks

first_week = df_clean["Week"].min()
last_week = df_clean["Week"].max()

df_clean = df_clean.loc[(df_clean["Week"] != first_week)
                        &(df_clean["Week"] != last_week)].copy()

In [129]:
# Tests

print(df_clean["Week"].nunique())
print(df_clean["Week"].min())
print(df_clean["Week"].max())

51
2010-12-06/2010-12-12
2011-11-28/2011-12-04


In [130]:
# define the full calendar weeks table + training / validation / test periods

complete_weeks = pd.period_range(
    start=first_week + 1,
    end=last_week - 1,
    freq="W"
)

train_weeks = complete_weeks[:34]
val_weeks = complete_weeks[34:43]
test_weeks = complete_weeks[43:]

In [131]:
# construction of the weekly demand tables for all the transactions + training period 

weekly_demand = (
    df_clean
    .groupby(["StockCode", "Week"])["Quantity"]
    .sum()
    .reset_index()
    .sort_values(["StockCode", "Week"])
    .reset_index(drop=True).copy()
)

weekly_demand_training = (df_clean.loc[df_clean["Week"].isin(train_weeks)]
    .groupby(["StockCode", "Week"])["Quantity"]
    .sum()
    .reset_index()
    .sort_values(["StockCode", "Week"])
    .reset_index(drop=True).copy()
                         )
# First .reset_index() simply makes "StockCode" and "Week" proper columns again, while the second one makes sure
# that the newly sorted tables have indexes in an increasing order

# .reset_index(): adds an index column 0, 1, 2, ... and the current one becomes an actual column
# .reset_index(drop=True): remove the current index column and adds an index column 0, 1, 2, ...

# Note that .groupby(["StockCode", "Week"]) gives the structure

# Product A
#    Week 1
#    Week 2
#    Week 3

# Product B
#    Week 1
#    Week 2
#    Week 3

# whilst .groupby(["Week", "StockCode"]) would give

# Week 1
#    Product A
#    Product B
#    Product C

# Week 2
#    Product A
#    Product B
#    Product C

In [132]:
# defining the tables of active weeks and checking which 

active_training_weeks = (
    weekly_demand_training
    .groupby("StockCode")
    .size()
    .reset_index(name="active_weeks").copy()
)

# .reset_undex(name="bla"): puts the result of .size() in a column named "bla", moves current index column to a normal column,
# and adds an index column 0, 1, 2, ...

In [133]:
# Selection of the product universe: products which appear sufficiently often
# We will only work on this product selection for our forecasting problem (also for validation and tests)

eligible_products = active_training_weeks.loc[
    active_training_weeks["active_weeks"] >= 28,
    "StockCode"
]

# syntax point: frame.loc[rows I keep, columns I keep] 

print(active_training_weeks.head(5))
print(eligible_products.head(5))

  StockCode  active_weeks
0     10002            17
1     10080             5
2     10120             8
3     10125            22
4     10133            28
4     10133
5     10135
8     15034
9     15036
10    15039
Name: StockCode, dtype: object


In [134]:
# Weekly table restricted to the product universe

weekly_demand_selected = weekly_demand.loc[weekly_demand["StockCode"].isin(eligible_products)].reset_index(drop=True)

print(weekly_demand_selected.head(5), weekly_demand_selected.shape)

  StockCode                   Week  Quantity
0     10133  2010-12-06/2010-12-12        14
1     10133  2010-12-13/2010-12-19        60
2     10133  2010-12-20/2010-12-26        16
3     10133  2011-01-03/2011-01-09        11
4     10133  2011-01-10/2011-01-16        67 (39954, 3)


In [135]:
# Two independent tables with stock codes and weeks for ALL THE CALENDAR YEAR 

products_df = pd.DataFrame({"StockCode": eligible_products})

weeks_df = pd.DataFrame({"Week": complete_weeks})

# and their 'cross-product'
product_week_grid = products_df.merge(weeks_df, how="cross")

In [136]:
# Weekly panel of weekly transactions for the selected products for the complete calendar year (52 weeks)
# note that we have constructed the weekly panel with the merge procedure below because
# some calendar weeks (just one) were not represented in df_clean (see df_clean["Week"].nunique() above)

weekly_panel = (
    product_week_grid
    .merge(
        weekly_demand_selected,
        on=["StockCode", "Week"],
        how="left",
        validate="one_to_one"
    )
    .sort_values(["StockCode", "Week"])
    .reset_index(drop=True)
)

print(weekly_panel.shape, weekly_panel.head(), weekly_panel["Week"].nunique())

(43628, 3)   StockCode                   Week  Quantity
0     10133  2010-12-06/2010-12-12      14.0
1     10133  2010-12-13/2010-12-19      60.0
2     10133  2010-12-20/2010-12-26      16.0
3     10133  2010-12-27/2011-01-02       NaN
4     10133  2011-01-03/2011-01-09      11.0 52


In [137]:
# Sanity checks: we've added 3674 (= 43628 - 39954) rows by adding the missing calendar week + weeks that correspond to a 
# sale but not necessarily from a product in eligible_products; see next cell

print("Eligible products:", len(eligible_products))
print("Observed selected product-weeks:", weekly_demand_selected.shape)
print("Full panel expected:", len(eligible_products) * len(complete_weeks))
print("Full panel actual:", weekly_panel.shape)

Eligible products: 839
Observed selected product-weeks: (39954, 3)
Full panel expected: 43628
Full panel actual: (43628, 3)


In [140]:
print(weekly_panel["Quantity"].isna().sum())

0


In [139]:
# replacing NaNs by 0s 

weekly_panel["Quantity"] = weekly_panel["Quantity"].fillna(0)

In [141]:
# Sanity checks

assert len(complete_weeks) == 52
assert len(train_weeks) + len(val_weeks) + len(test_weeks) == 52
assert eligible_products.is_unique

assert weekly_panel.shape[0] == len(eligible_products) * len(complete_weeks)

assert not weekly_panel.duplicated(["StockCode", "Week"]).any()

# checks that no product-week pair appears more than once

assert weekly_panel.groupby("StockCode")["Week"].nunique().eq(52).all()

# Checks that every product has exactly 52 distinct weeks:
# 1. Group rows by product.
# 2. Count each product’s distinct weeks.
# 3. .eq(52) checks whether each count equals 52.
# 4. .all() requires this to be true for every product. 


assert weekly_panel["Quantity"].notna().all()

# Checks that every Quantity value is present. This confirms that all missing values were successfully replaced with zero.

assert weekly_panel["Quantity"].ge(0).all()

# Checks that every Quantity value is present. This confirms that all missing values were successfully replaced with zero.

In [145]:
# Definition of lag features 

weekly_panel["lag_1"] = (
    weekly_panel
    .groupby("StockCode")["Quantity"]
    .shift(1)
)

weekly_panel["lag_2"] = (
    weekly_panel
    .groupby("StockCode")["Quantity"]
    .shift(2)
)

weekly_panel["lag_3"] = (
    weekly_panel
    .groupby("StockCode")["Quantity"]
    .shift(3)
)

weekly_panel["lag_4"] = (
    weekly_panel
    .groupby("StockCode")["Quantity"]
    .shift(4)
)

print(weekly_panel.head())

  StockCode                   Week  Quantity  lag_1  lag_2  lag_3  lag_4
0     10133  2010-12-06/2010-12-12      14.0    NaN    NaN    NaN    NaN
1     10133  2010-12-13/2010-12-19      60.0   14.0    NaN    NaN    NaN
2     10133  2010-12-20/2010-12-26      16.0   60.0   14.0    NaN    NaN
3     10133  2010-12-27/2011-01-02       0.0   16.0   60.0   14.0    NaN
4     10133  2011-01-03/2011-01-09      11.0    0.0   16.0   60.0   14.0


In [148]:
# removing rows corresponding  to NaNs from the weekly panel

weekly_panel_lag = weekly_panel.loc[
    weekly_panel["lag_4"].notna()
].reset_index(drop=True).copy()

print(weekly_panel_lag.head(5))

  StockCode                   Week  Quantity  lag_1  lag_2  lag_3  lag_4
0     10133  2011-01-03/2011-01-09      11.0    0.0   16.0   60.0   14.0
1     10133  2011-01-10/2011-01-16      67.0   11.0    0.0   16.0   60.0
2     10133  2011-01-17/2011-01-23      92.0   67.0   11.0    0.0   16.0
3     10133  2011-01-24/2011-01-30      10.0   92.0   67.0   11.0    0.0
4     10133  2011-01-31/2011-02-06       0.0   10.0   92.0   67.0   11.0


In [149]:
# Defintion of the different data sets

train_data = weekly_panel_lag.loc[
    weekly_panel_lag["Week"].isin(train_weeks)
].copy()

val_data = weekly_panel_lag.loc[
    weekly_panel_lag["Week"].isin(val_weeks)
].copy()

test_data = weekly_panel_lag.loc[
    weekly_panel_lag["Week"].isin(test_weeks)
].copy()

In [150]:
# Definition of inputs and outputs for training / val / test periods

features = ["lag_1", "lag_2", "lag_3", "lag_4"]

X_train = train_data[features]
y_train = train_data["Quantity"]

X_val = val_data[features]
y_val = val_data["Quantity"]

X_test = test_data[features]
y_test = test_data["Quantity"]

In [152]:
# Sanity check: zero missing values from all feature matrices

print(X_train.shape, y_train.shape)
print(X_val.shape, y_val.shape)
print(X_test.shape, y_test.shape)

print(X_train.isna().sum())
print(X_val.isna().sum())
print(X_test.isna().sum())

(25170, 4) (25170,)
(7551, 4) (7551,)
(7551, 4) (7551,)
lag_1    0
lag_2    0
lag_3    0
lag_4    0
dtype: int64
lag_1    0
lag_2    0
lag_3    0
lag_4    0
dtype: int64
lag_1    0
lag_2    0
lag_3    0
lag_4    0
dtype: int64
